In [ ]:
pip install selenium #Run only once

In [ ]:
#Just change the name of state to your choice, where UTTAR PRADESH is written 
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import csv
import time

# Initialize driver
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 20)

driver.get("https://saras.cbse.gov.in/saras/AffiliatedList/ListOfSchdirReport")

state_wise_radio = wait.until(
    EC.element_to_be_clickable((By.ID, "SearchMainRadioState_wise"))
)
driver.execute_script("arguments[0].click();", state_wise_radio)
time.sleep(1)

state_dropdown = wait.until(
    EC.presence_of_element_located((By.ID, "State"))
)
Select(state_dropdown).select_by_visible_text("UTTAR PRADESH") #Change the state name here, keep the case same

search_button = wait.until(
    EC.element_to_be_clickable((By.XPATH, "//input[@value='Search']"))
)
driver.execute_script("arguments[0].click();", search_button)

wait.until(EC.presence_of_element_located((By.ID, "myTable")))
wait.until(
    EC.presence_of_element_located(
        (By.XPATH, "//table[@id='myTable']//tbody/tr")
    )
)

wait.until(
    EC.presence_of_element_located(
        (By.XPATH, "//table[@id='myTable']//tbody/tr")
    )
)

data = []

while True:
    rows = driver.find_elements(
        By.XPATH, "//table[@id='myTable']//tbody/tr"
    )

    for row in rows:
        cols = [c.text.strip() for c in row.find_elements(By.TAG_NAME, "td")]
        if len(cols) == 7:
            data.append(cols)

    try:
        next_button = driver.find_element(By.ID, "myTable_next")

        if "disabled" in next_button.get_attribute("class"):
            break

        driver.execute_script("arguments[0].click();", next_button)
        time.sleep(1.5)

    except:
        break


with open("cbse_uttar_pradesh_schools.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow([
        "S NO",
        "AFF. NO & SCHOOL_CODE",
        "STATE AND DISTRICT",
        "STATUS",
        "SCHOOL AND HEAD_NAME",
        "ADDRESS",
        "DETAILS"
    ])
    writer.writerows(data)

print(f"Total schools scraped: {len(data)}")

driver.quit()


In [ ]:
pip install openpyxl

In [ ]:
# For details page
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import csv
import time

# ---------------- CONFIG ----------------
INPUT_EXCEL = "aff_up.xlsx"
OUTPUT_CSV = "details_up.csv"
AFF_COL = "AFF_COL"  #change if column name is different
BASE_URL = "https://saras.cbse.gov.in/saras/AffiliatedList/AfflicationDetails/"
# ----------------------------------------

# Load affiliation codes
df = pd.read_excel(INPUT_EXCEL)
affiliation_codes = df[AFF_COL].astype(str).tolist()

# Start Selenium
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 20)

results = []

for aff_code in affiliation_codes:
    url = BASE_URL + aff_code
    driver.get(url)

    try:
        # Wait for details table to load
        wait.until(
            EC.presence_of_element_located(
                (By.XPATH, "//table//tr")
            )
        )

        def get_value(label):
            try:
                return driver.find_element(
                    By.XPATH,
                    f"//td[normalize-space()='{label}']/following-sibling::td"
                ).text.strip()
            except:
                return ""

        record = {
            "Affiliation_Number": aff_code,
            "Pin_Code": get_value("Pin Code"),
            "Year_of_Foundation": get_value("Year of Foundation"),
            "Date_of_First_Opening": get_value("Date of First Opening of School"),
            "School_Type": get_value("School Type"),
            "Trust_or_Society_Name": get_value(
                "Name of Trust/ Society/ Managing Committee"
            ),
            "Website": get_value("Website")
        }

        results.append(record)
        print(f"✔ Scraped {aff_code}")

    except Exception as e:
        print(f"✖ Failed for {aff_code}")
        results.append({
            "Affiliation_Number": aff_code,
            "Pin_Code": "",
            "Year_of_Foundation": "",
            "Date_of_First_Opening": "",
            "School_Type": "",
            "Trust_or_Society_Name": "",
            "Website": ""
        })

    time.sleep(1)  # polite delay

driver.quit()

# Save to CSV
pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"\n✅ Completed. Data saved to {OUTPUT_CSV}")